In [2]:
!pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.2/376.2 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.8/494.8 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 111.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 91.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 864.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Colab Link: https://colab.research.google.com/drive/12Z-E6sBmuNuf2Ki2mSI8ZNd9cAsJPOqn?authuser=6#scrollTo=VIxlIGO7HpYy

In [1]:
import torch
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import TrainingArguments, AutoTokenizer, AutoModelForCausalLM
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM, SFTConfig

In [2]:
def load_model_and_tokenizer(model_name, use_gpu=False):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)

    if use_gpu:
        model.to("cuda")

    if not tokenizer.chat_template:
        tokenizer.chat_template = """{% for message in messages %}
            {% if message['role'] == 'system' %}System: {{ message['content'] }}\n
            {% elif message['role'] == 'user' %}User: {{ message['content'] }}\n
            {% elif message['role'] == 'assistant' %}Assistant: {{ message['content'] }} <|endoftext|>
            {% endif %}
        {% endfor %}"""

    if not tokenizer.pad_token:
        tokenizer.pad_token = tokenizer.eos_token

    return model, tokenizer

In [3]:
def display_dataset(dataset):
    rows = []
    for i in range(3):
        example = dataset[i]
        user_msg = next(m['content'] for m in example['messages'] if m['role'] == 'user')
        assistant_msg = next(m['content'] for m in example['messages'] if m['role'] == 'assistant')
        rows.append({'User Prompt': user_msg, 'Assistant Response': assistant_msg})

    df = pd.DataFrame(rows)
    pd.set_option('display.max_colwidth', None)
    display(df)

In [4]:
def generate_responses(model, tokenizer, user_message, system_message=None, max_new_tokens=100):
    messages = []
    if system_message:
        messages.append({"role": "system", "content": system_message})
    messages.append({"role": "user", "content": user_message})

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    input_len = inputs["input_ids"].shape[1]
    generated_ids = outputs[0][input_len:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    return response

In [5]:
def test_model_with_questions(model, tokenizer, questions, system_message=None, title="Model Output"):
    print(f"\n=== {title} ===")
    for i, question in enumerate(questions, 1):
        response = generate_responses(model, tokenizer, question, system_message)
        print(f"\nModel Input {i}:\n{question}\nModel Output {i}:\n{response}\n")

In [6]:
!git clone https://huggingface.co/HuggingFaceTB/SmolLM2-135M

Cloning into 'SmolLM2-135M'...
remote: Enumerating objects: 37, done.
remote: Counting objects: 100% (34/34), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 37 (delta 14), reused 0 (delta 0), pack-reused 3 (from 1)
Unpacking objects: 100% (37/37), 1.10 MiB | 5.42 MiB/s, done.


In [7]:
USE_GPU = True

questions = [
    "Who is Pankaj Kumar?",
    "Where did Pankaj Studied?",
    "What is the background of Pankaj?"
]

model, tokenizer = load_model_and_tokenizer("./SmolLM2-135M", USE_GPU)
test_model_with_questions(model, tokenizer, questions, title="Base Model (Before SFT) Output")
del model, tokenizer


=== Base Model (Before SFT) Output ===

Model Input 1:
Who is Pankaj Kumar?
Model Output 1:
Pankaj Kumar is a 20-year-old student from Mumbai. He is a student of the University of Mumbai. He is a student of the University of Mumbai. He is a student of the University of Mumbai. He is a student of the University of Mumbai. He is a student of the University of Mumbai. He is a student of the University of Mumbai. He is a student of the University of Mumbai. He is a student of the University of Mumbai. He is


Model Input 2:
Where did Pankaj Studied?
Model Output 2:
Pankaj Studied: Where did Pankaj Studied?

Pankaj Studied: Where did Pankaj Studied?

Pankaj Studied: Where did Pankaj Studied?

Pankaj Studied: Where did Pankaj Studied?

Pankaj Studied: Where did Pankaj Studied?

Pankaj Studied: Where did Pankaj Studied?

Pankaj


Model Input 3:
What is the background of Pankaj?
Model Output 3:
Pankaj is a 20-year-old student from Mumbai. He is a student of the University of Mumbai. He is a s

In [8]:
# Expanding the dataset to 50 Q&A pairs based on resume content

qa_pairs = [
    # Education
    ("Where did you complete your B.Tech?", "I completed my Bachelor of Technology at IIT Guwahati."),
    ("What was your CGPA in college?", "My CGPA at IIT Guwahati was 7.87 out of 10."),
    ("Which school did you attend for your 12th grade?", "I studied at St. Xavier's Jr./Sr. School for my Intermediate of Science."),
    ("What percentage did you score in 12th?", "I scored 93% in my 12th grade."),
    ("Where did you complete your 10th grade?", "I completed my 10th from St. Xavier's Jr./Sr. School."),
    ("What was your CGPA in 10th?", "I scored a perfect CGPA of 10 in my 10th grade."),

    # Experience at Meta
    ("What was your role at Meta?", "I worked as an AI Delivery Manager at Meta through Turing."),
    ("How many developers did you lead at Meta?", "I led large-scale teams of 300+ developers."),
    ("What kind of projects did you manage at Meta?", "I managed projects like LLaMA4-RLHF-Coding and Pretrain-Benchmarking."),
    ("How did you improve data quality at Meta?", "I developed quality control tools, reducing data wastage from 40% to 20% and increasing review coverage to 100%."),
    ("How fast did you scale teams at Meta?", "I scaled teams from 50 to over 250 members in under a week."),
    ("What tools did you develop at Meta?", "I developed a duplicate task checker and an AI-powered auto-reviewer."),
    ("What was your role from May to Sept 2024?", "I was a Team Lead and Senior Python Developer, improving coding capability of Llama3.2."),

    # Experience at OpenAI
    ("What was your role at OpenAI in 2023-2024?", "I worked as a Pod Lead and Python Developer."),
    ("What was your responsibility as a Pod Lead at OpenAI?", "I led a team of 7 prompt engineers for ChatGPT-4.5."),
    ("What kind of data did you work on at OpenAI?", "I worked on coding datasets for RLHF, SFT, and function/tool calling pipelines."),
    ("What was your role at OpenAI in 2022?", "I was a Prompt Engineer creating high-quality datasets for ChatGPT models."),

    # Valuence Technologies
    ("Where did you work as a freelancer?", "I worked with Valuence Technologies."),
    ("What was your contribution to helpmeee KEIKO?", "I implemented RAG, finetuned Japanese GPT model, and integrated ChatGPT API."),
    ("Where did you deploy the custom GPT model?", "I deployed it on AWS."),

    # Internship
    ("Where did you intern during college?", "I interned at National Tsing Hua University in Taiwan."),
    ("What research did you do during your internship?", "I studied image aesthetic assessment using neural networks."),

    # Skills
    ("Which programming languages do you know?", "I am skilled in Python and C++."),
    ("Which AI tools have you worked with?", "I have worked with PyTorch, TensorFlow, and SkLearn."),
    ("What API frameworks do you know?", "I have experience with FastAPI and Flask."),
    ("Do you have experience with databases?", "Yes, I have worked with MySQL."),
    ("Which cloud platforms have you used?", "I have worked with AWS and SageMaker."),
    ("Do you have experience with containerization?", "Yes, I have used Docker for containerization."),
    ("What are your interpersonal skills?", "I possess leadership qualities and a positive attitude."),
    ("Which languages do you speak?", "I speak Hindi, English, and some Japanese."),

    # Awards
    ("Have you received any awards?", "Yes, I was a finalist in the AI Hackathon by C-DAC, Nvidia, and ATOS."),
    ("What award did you win in GASE 2019?", "I won the Popular Award in the GASE 2019 Program by MOST."),

    # Courses
    ("Which AI courses have you taken?", "I completed IBM Data Science Professional and Deep Learning Specialization."),
    ("What DevOps knowledge do you have?", "I completed a DevOps course from beginner to advanced level."),
    ("Have you studied data structures?", "Yes, I completed a bootcamp on Data Structures and Algorithms."),
    ("Have you taken any NLP courses?", "Yes, I took a course on NLP with Transformers."),
    ("Which AWS services have you used?", "I have used AWS SageMaker and AWS Lambda."),
    ("Have you taken any REST API courses?", "Yes, I studied designing RESTful APIs."),
    ("Have you studied LangChain?", "Yes, I have studied LangChain and Agentic RAG."),
    ("Which tools do you know for prompt engineering?", "I am experienced in prompt engineering with LangGraph, MCP, A2A, and ACP."),

    # Projects and Tools
    ("What is your GitHub username?", "My GitHub username is ivrschool."),
    ("Do you write technical content?", "Yes, I write on Medium about AI fundamentals."),
    ("Do you use Hugging Face?", "Yes, I use Hugging Face datasets and transformers."),
    ("Have you worked on RLHF?", "Yes, I worked extensively on RLHF projects at Meta and OpenAI."),
    ("Have you performed SFT on models?", "Yes, I have fine-tuned models using supervised fine-tuning techniques."),
    ("Have you done any work on tool calling?", "Yes, I contributed to function and tool calling pipelines for ChatGPT."),
]

# Convert to dataset format
dataset = []
for q, a in qa_pairs:
    dataset.append({
        "messages": [
            {"role": "user", "content": q},
            {"role": "assistant", "content": a}
        ]
    })

# Display first 5 to user
rows = []
for i in range(5):
    example = dataset[i]
    user_msg = next(m['content'] for m in example['messages'] if m['role'] == 'user')
    assistant_msg = next(m['content'] for m in example['messages'] if m['role'] == 'assistant')
    rows.append({'User Prompt': user_msg, 'Assistant Response': assistant_msg})

df = pd.DataFrame(rows)
# import ace_tools as tools; tools.display_dataframe_to_user(name="50 Resume Q&A Pairs (Preview)", dataframe=df)


In [9]:
df.head()

,User Prompt,Assistant Response
0,Where did you complete your B.Tech?,I completed my Bachelor of Technology at IIT G...
1,What was your CGPA in college?,My CGPA at IIT Guwahati was 7.87 out of 10.
2,Which school did you attend for your 12th grade?,I studied at St. Xavier's Jr./Sr. School for m...
3,What percentage did you score in 12th?,I scored 93% in my 12th grade.
4,Where did you complete your 10th grade?,I completed my 10th from St. Xavier's Jr./Sr. ...


In [10]:
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
import os

# Split the dataset
train_data, temp_data = train_test_split(dataset, test_size=0.2, random_state=42)
val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42)

# Create DatasetDict
dataset_dict = DatasetDict({
    "train": Dataset.from_list(train_data),
    "validation": Dataset.from_list(val_data),
    "test": Dataset.from_list(test_data)
})

# Save to disk in Hugging Face format
save_path = "./data/myDataset1"
dataset_dict.save_to_disk(save_path)


Saving the dataset (0/1 shards):   0%|          | 0/36 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5 [00:00<?, ? examples/s]

In [17]:
from datasets import load_from_disk

dataset = load_from_disk("./data/myDataset1")
train_dataset = dataset["train"]
validation_dataset = dataset["validation"]

display_dataset(train_dataset)

,User Prompt,Assistant Response
0,What was your role at Meta?,I worked as an AI Delivery Manager at Meta through Turing.
1,What API frameworks do you know?,I have experience with FastAPI and Flask.
2,Which AI courses have you taken?,I completed IBM Data Science Professional and Deep Learning Specialization.


In [61]:
model_name = "./SmolLM2-135M"
model, tokenizer = load_model_and_tokenizer(model_name, USE_GPU)

In [62]:
sft_config = SFTConfig(
    learning_rate=8e-5,
    report_to="none", # disable logging to W&B
    num_train_epochs=10,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    logging_steps=2,
    logging_strategy="steps",
    eval_strategy="epoch",                   # evaluate at end of each epoch
    save_strategy="epoch",                   # save checkpoint at end of each epoch
    save_total_limit=1,                      # keep only the best/latest model
    load_best_model_at_end=True,             # load best model according to eval loss
    metric_for_best_model="eval_loss",       # use eval loss for best model selection
    greater_is_better=False,                 # lower eval_loss is better
    output_dir="./checkpoints"               # directory to save checkpoints
)


In [63]:
sft_trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    processing_class=tokenizer,
)

sft_trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,2.784258
2,3.145300,2.418189
3,3.145300,2.291856
4,2.507500,1.883099
5,2.507500,1.785080
6,1.840900,1.719926
7,1.840900,1.658811
8,1.655200,1.622244
9,1.655200,1.597691
10,1.538700,1.589749


There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=10, training_loss=2.137508916854858, metrics={'train_runtime': 230.853, 'train_samples_per_second': 1.559, 'train_steps_per_second': 0.043, 'total_flos': 9332097678720.0, 'train_loss': 2.137508916854858})

In [68]:
model = AutoModelForCausalLM.from_pretrained("./checkpoints/checkpoint-10")  # or latest step
tokenizer = AutoTokenizer.from_pretrained("./checkpoints/checkpoint-10")


In [65]:


questions = [
    "Who is Pankaj Kumar?",
    "Where did Pankaj Studied?",
    "What is the background of Pankaj?"
]

test_model_with_questions(model, tokenizer, questions, title="Base Model (After SFT) Output")



=== Base Model (Before SFT) Output ===

Model Input 1:
Who is Pankaj Kumar?
Model Output 1:
1. What is your area of expertise?

I am a Data Scientist at IIT Guwahati. I have worked on various projects in Machine Learning, Data Science, and Data Engineering. I have also worked on Data Science and Data Engineering projects at IIT Guwahati. I have a Bachelor of Science in Data Science from IIT Guwahati. I have a Master of Science in Data Science from IIT Guwahati. I have worked on various projects


Model Input 2:
Where did Pankaj Studied?
Model Output 2:
1. What was his major research area?

Pankaj Studied:

He studied the role of the brain in the development of autism. He also studied the role of the brain in the development of schizophrenia. He also studied the role of the brain in the development of autism.

2. What was your major research project?

Pankaj Studied:

He studied the role of the brain in the development of autism. He also studied the role of the


Model Input 3:
What is

In [71]:
from transformers import EarlyStoppingCallback
sft_config = SFTConfig(
    learning_rate=8e-5,
    report_to="none", # disable logging to W&B
    num_train_epochs=50,
    per_device_train_batch_size=5,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    logging_steps=2,
    logging_strategy="steps",
    eval_strategy="epoch",                   # evaluate at end of each epoch
    save_strategy="epoch",                   # save checkpoint at end of each epoch
    save_total_limit=1,                      # keep only the best/latest model
    load_best_model_at_end=True,             # load best model according to eval loss
    metric_for_best_model="eval_loss",       # use eval loss for best model selection
    greater_is_better=False,                 # lower eval_loss is better
    output_dir="./checkpoints"               # directory to save checkpoints
)

# Instantiate early stopping callback
early_stopping_callback = EarlyStoppingCallback(
    early_stopping_patience=2  # Stop if no improvement for 2 evals (epochs)
)


In [72]:
sft_trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    processing_class=tokenizer,
    callbacks=[early_stopping_callback]
)

sft_trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,1.466817
2,0.226200,1.404416
3,0.226200,1.440710
4,0.174900,1.493215


There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=4, training_loss=0.20052778720855713, metrics={'train_runtime': 154.2234, 'train_samples_per_second': 11.671, 'train_steps_per_second': 0.324, 'total_flos': 3688233619968.0, 'train_loss': 0.20052778720855713})

In [75]:
model = AutoModelForCausalLM.from_pretrained("./checkpoints/checkpoint-2")  # or latest step
tokenizer = AutoTokenizer.from_pretrained("./checkpoints/checkpoint-2")


questions = [
    "Who is Pankaj Kumar?",
    "Where did Pankaj Studied?",
    "What is the background of Pankaj?"
]

test_model_with_questions(model, tokenizer, questions, title="Base Model (After SFT) Output")


=== Base Model (After SFT) Output ===

Model Input 1:
Who is Pankaj Kumar?
Model Output 1:
Assistant: I am Pankaj Kumar.


Model Input 2:
Where did Pankaj Studied?
Model Output 2:
Assistant: I studied at IIT Guwahati.


Model Input 3:
What is the background of Pankaj?
Model Output 3:
Assistant: I completed a bootcamp on Data Science and Machine Learning by IIT Guwahati.



In [78]:
def evaluate_model_on_test_set(model, tokenizer, test_dataset, num_samples=10):
    print("\n=== Model Evaluation on Test Set ===\n")
    for i in range(min(num_samples, len(test_dataset))):
        sample = test_dataset[i]
        messages = sample["messages"]
        user_msg = next(m["content"] for m in messages if m["role"] == "user")
        gold_msg = next(m["content"] for m in messages if m["role"] == "assistant")

        model_response = generate_responses(model, tokenizer, user_msg)

        print(f"\n--- Sample {i + 1} ---")
        print(f"User Prompt       : {user_msg}")
        print(f"Original Response : {gold_msg}")
        print(f"Model Response    : {model_response}")

In [79]:
test_dataset = dataset["test"]

# Put model in eval mode and move to device
model.eval()
model.to("cuda" if torch.cuda.is_available() else "cpu")

# Evaluate
evaluate_model_on_test_set(model, tokenizer, test_dataset, num_samples=10)


=== Model Evaluation on Test Set ===


--- Sample 1 ---
User Prompt       : What kind of projects did you manage at Meta?
Original Response : I managed projects like LLaMA4-RLHF-Coding and Pretrain-Benchmarking.
Model Response    : Assistant: I worked on collaborative writing projects with 500+ authors.

--- Sample 2 ---
User Prompt       : Do you have experience with databases?
Original Response : Yes, I have worked with MySQL.
Model Response    : Assistant: Yes, I have used HBase and Hive.

--- Sample 3 ---
User Prompt       : Do you write technical content?
Original Response : Yes, I write on Medium about AI fundamentals.
Model Response    : Assistant: Yes, I write technical documentation for various industries.

--- Sample 4 ---
User Prompt       : Which tools do you know for prompt engineering?
Original Response : I am experienced in prompt engineering with LangGraph, MCP, A2A, and ACP.
Model Response    : Assistant: I have experience with TDD and test-driven development.

--- Sa